### In this first example, we will explain the functionality of the MOMRollRateTable class

### MOMRollRateTable

In Application and Behavioural Scorecards, most of the time we don't have an exact definition on the bad customers.
So what we need to do is to decide who they are. Most of the time we decide that based on their delinquency status (e.g. 1-month delinquent, 2-month deliqnuent, etc) from month i to month i+1 (suppose September to October).

Firstly, let's see our data. In the tests folder there is a directory simulation_data. We pick 2 files representing month i and month i+1

In [ ]:
import polars as pl

data_i = pl.scan_csv("../tests/simulation_data/test_sample_0.csv").collect()
data_i_1 = pl.scan_csv("../tests/simulation_data/test_sample_1.csv").collect()

In [ ]:
data_i.head()

In [ ]:
data_i_1.head()

We can see that there are 8 columns:

**id**: Account id, the unique key of the dataset. \
**delq**: The delinquency of each account. \
**bin_ind_1**: A binary indicator. \
**bin_ind_2**: A binary indicator. \
**Open, Active, Deactive, Closed**: Indicates if the account is open, active, deactive or closed in that month. 

The last 6 columns are not of any use to us for this example.

So in order to get the roll rates from test_sample_0 to test_sample_1 we instanciate a MOMRllRateTable object and passing in the arguments needed, the unique key column of the 2 files **(should have the same name)**, the column which indicates the delinquency **(should have the same name)**, the paths to the 2 files (path_i for month i and path_i_1 for month i+1), the max deliqnuency we want to track and finally if we want to inlcude any binary indicators.

Then we call mehtod build() on the object to calculates the roll rate table.

In [ ]:
from roll_rate_analysis import MOMRollRateTable

table = MOMRollRateTable(
    unique_key_col="id",
    delinquency_col="delq",
    month_i="../tests/simulation_data/test_sample_0.csv",
    month_i_plus_1="../tests/simulation_data/test_sample_1.csv",
    max_delq=6,
)

In [ ]:
table.compute()

So that does that table tell us? The sum of accounts in row **n** indicates the number of accounts that were **n** cycle delinquent in month i. The sum of accounts in column **k** indicates the number of accounts that were **k** cycle delinquent in month i+1. So the value at **(2,3)** shows that **2308** accounts that were 1 cycle delinquent on month i, are 2 cycle delinquent on month i+1, which means they didn't pay the installment in time.

Then we can call reduce to get a more high level view of that table like below.

In [ ]:
table.reduce()

The table above gives us higher level info about the perventages of accounts in each bucket. \
**roll_down**: they paid their installment and at least one previous installment they owed \
**stable**: they just paid current installment (or remained in the 6+ deliqnuency bucket) \
**roll_up**: they didn't pay their current installment and their status chenged to worse

In most cases, the bucket where we see that the roll_up percentage is higher than 50% is the point were we decide **accounts from that bucket and below classify as bad accounts** (i.e. 4, 5, 6+ in our case).

In general, we compute the roll rates for a larger period of time (e.g. a year) and sum them up and go on from that. But for the purpose of this example its not necessary.

# Miscellaneous

If we wanted to track delinquencies bigger than 6 then we could change the max_delq argument at initialization:

In [ ]:
table = MOMRollRateTable(
    unique_key_col="id",
    delinquency_col="delq",
    month_i="../tests/simulation_data/test_sample_0.csv",
    month_i_plus_1="../tests/simulation_data/test_sample_1.csv",
    max_delq=7,
)
table.compute()

In [ ]:
table.reduce()